# Zain Jordan Customer 360 AI Workshop  
## Class 8: Capstone Build Sprint

### Class Goal

In Class 7, each team created a capstone blueprint.

In Class 8, teams will start building their working capstone prototype.

This notebook is a flexible **capstone starter kit**.

Teams can choose one or more build tracks:

| Track | Best For |
|---|---|
| Track A | SQL Agent projects |
| Track B | Tools-based customer-care projects |
| Track C | RAG recommendation projects |
| Track D | Multi-agent projects |
| Track E | MCP-enhanced projects |

---

## Class 8 Outcome

By the end of this build sprint, each team should have:

1. A working notebook or prototype.
2. One successful demo prompt.
3. A database-backed answer.
4. One AI-generated recommendation or summary.
5. A clear 5-minute demo flow.


# 1. Build Sprint Rules

Keep the project realistic.

## Must Have

1. One clear business problem.
2. One target user.
3. One working prompt.
4. Database-backed output.
5. Business recommendation.
6. Clear demo flow.

## Nice to Have

- RAG recommendation
- Multi-agent architecture
- MCP tools
- Gradio UI
- Customer-care message
- Executive summary

## Trainer Reminder

Do not let teams build everything.  
Help them build the smallest useful demo first.


# 2. Install Required Packages


In [ ]:
%pip install -q -U langchain langchain-openai langchain-community langchain-text-splitters langchain-mcp-adapters "mcp[cli]" fastmcp pandas sqlalchemy


# 3. Import Libraries


In [ ]:
import os
import sqlite3
import pandas as pd
from pathlib import Path
from datetime import datetime
from collections import Counter

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

print("Libraries imported successfully.")


# 4. Set OpenAI API Key

In Google Colab:

1. Click the key icon on the left sidebar.
2. Add a secret named `OPENAI_API_KEY`.
3. Paste your OpenAI API key.
4. Enable notebook access for the secret.


In [ ]:
try:
    from google.colab import userdata
    openai_key = userdata.get("OPENAI_API_KEY")
    if openai_key:
        os.environ["OPENAI_API_KEY"] = openai_key
        print("OpenAI API key loaded from Colab Secrets.")
    else:
        print("OPENAI_API_KEY not found in Colab Secrets.")
except Exception:
    print("Not running in Google Colab, or Colab Secrets not available.")

if not os.environ.get("OPENAI_API_KEY"):
    print("Warning: OPENAI_API_KEY is not set. Agent cells will not run until it is configured.")
else:
    print("OPENAI_API_KEY is available.")


# 5. Upload or Locate the Zain Jordan Database

Upload the same database file used in previous classes:

`zain_customer_360_ai_demo.db`


In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    print("Uploaded files:", list(uploaded.keys()))
except Exception:
    print("Google Colab upload is not available here. If running locally, place the .db file in the notebook folder.")


# 6. Connect to SQLite Database


In [ ]:
db_files = [file for file in os.listdir() if file.endswith(".db")]

if db_files:
    DB_PATH = db_files[0]
else:
    DB_PATH = "zain_customer_360_ai_demo.db"

db_path_obj = Path(DB_PATH).resolve()

print("Database path:", db_path_obj)
print("File exists:", db_path_obj.exists())

conn = sqlite3.connect(str(db_path_obj), check_same_thread=False)

tables_df = pd.read_sql_query("""
SELECT name 
FROM sqlite_master 
WHERE type = 'table'
ORDER BY name;
""", conn)

print("Number of tables:", len(tables_df))
tables_df


# 7. Database Row Counts


In [ ]:
table_counts = []

for table_name in tables_df["name"]:
    count_query = f"SELECT COUNT(*) AS row_count FROM {table_name}"
    count = pd.read_sql_query(count_query, conn)["row_count"][0]
    table_counts.append({
        "table_name": table_name,
        "row_count": count
    })

table_counts_df = pd.DataFrame(table_counts).sort_values("row_count", ascending=False)
table_counts_df


# 8. Team Capstone Configuration

Teams should update this section based on their Class 7 blueprint.


In [ ]:
team_config = {
    "team_name": "Team Name Here",
    "project_title": "Churn Rescue Assistant",
    "business_problem": "Retention teams need to identify high-risk, high-value customers and recommend actions.",
    "target_user": "Retention team / customer success manager",
    "main_demo_prompt": "Find high-value customers at high churn risk and recommend retention actions.",
    "selected_track": "Track A + Track B + Track C",
    "database_tables": [
        "customers",
        "customer_churn_scores",
        "customer_value_segments",
        "complaints",
        "support_interactions",
        "plans",
        "campaigns"
    ]
}

team_config


# 9. Initialize the Language Model


In [ ]:
from langchain.chat_models import init_chat_model

MODEL_NAME = "gpt-4.1-mini"

llm = init_chat_model(
    MODEL_NAME,
    model_provider="openai",
    temperature=0
)

print("LLM initialized:", MODEL_NAME)


# 10. Shared Helper Functions


In [ ]:
def df_to_text(df, max_rows=10):
    """Convert a pandas DataFrame into readable text."""
    if df is None or df.empty:
        return "No records found."
    return df.head(max_rows).to_string(index=False)


def extract_final_text(result):
    """Extract readable final text from a LangChain agent result."""
    last_message = result["messages"][-1]

    if hasattr(last_message, "content") and isinstance(last_message.content, str):
        return last_message.content

    if hasattr(last_message, "content_blocks"):
        parts = []
        for block in last_message.content_blocks:
            if isinstance(block, dict):
                if "text" in block:
                    parts.append(block["text"])
                elif "content" in block:
                    parts.append(str(block["content"]))
                else:
                    parts.append(str(block))
            else:
                parts.append(str(block))
        return "\n".join(parts)

    return str(last_message)


def run_agent(agent, question: str):
    result = agent.invoke({
        "messages": [
            {"role": "user", "content": question}
        ]
    })
    return extract_final_text(result)


def save_text_report(filename: str, content: str):
    path = Path(filename)
    path.write_text(content, encoding="utf-8")
    print(f"Saved report to: {path.resolve()}")
    return path


# 11. Build Track Selection Guide

| Track | Use When |
|---|---|
| Track A: SQL Agent | You need rankings, counts, summaries, insights |
| Track B: Tools Agent | You need customer-specific lookup |
| Track C: RAG | You need plan, campaign, add-on, or complaint recommendations |
| Track D: Multi-Agent | You need multiple specialists working together |
| Track E: MCP | You want enterprise-style tool exposure |

Start with one working track first.


# Track A: SQL Agent Starter

Use this if your project needs churn analysis, revenue insights, campaign performance, complaint analytics, executive KPI summaries, or network impact analysis.


In [ ]:
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain.agents import create_agent

db_uri = f"sqlite:///{db_path_obj}"
db = SQLDatabase.from_uri(db_uri)

sql_toolkit = SQLDatabaseToolkit(db=db, llm=llm)
sql_tools = sql_toolkit.get_tools()

sql_agent_prompt = """
You are a telecom business intelligence SQL Agent for Zain Jordan.

Rules:
- Use only SELECT queries.
- Never modify the database.
- Inspect schema before complex queries.
- Limit results to 10 rows unless asked otherwise.
- Explain the business meaning of results.
- Do not guess facts outside the database.

Useful table guidance:
- Churn: customers, customer_churn_scores
- Value/revenue: customer_value_segments, invoices, payments, transactions
- Complaints/support: complaints, support_interactions
- Campaigns: campaigns, customer_campaign_responses
- Network: network_towers, network_events
"""

sql_agent = create_agent(
    model=llm,
    tools=sql_tools,
    system_prompt=sql_agent_prompt,
)

print("Track A SQL Agent ready.")


## Track A Demo Question


In [ ]:
sql_demo_question = "Which cities have the most high-risk churn customers? Show the top 5 and explain the business meaning."

print(run_agent(sql_agent, sql_demo_question))


# Track B: Tools-Based Customer Agent Starter

Use this if your project needs customer-specific lookup.


In [ ]:
def fetch_customer_profile(customer_id: int) -> str:
    query = """
    SELECT 
        c.customer_id,
        c.full_name,
        c.gender,
        c.age_group,
        c.city,
        c.governorate,
        c.customer_type,
        c.customer_segment,
        c.preferred_language,
        c.status AS customer_status,
        a.account_type,
        a.account_status,
        a.credit_limit_jod
    FROM customers c
    LEFT JOIN accounts a
        ON c.customer_id = a.customer_id
    WHERE c.customer_id = ?;
    """
    df = pd.read_sql_query(query, conn, params=(customer_id,))
    return df_to_text(df)


def fetch_customer_plan(customer_id: int) -> str:
    query = """
    SELECT 
        c.customer_id,
        c.full_name,
        s.subscription_id,
        s.msisdn,
        s.service_type,
        s.status AS subscription_status,
        p.plan_name,
        p.plan_category,
        p.monthly_fee_jod,
        p.data_allowance_gb,
        p.local_minutes,
        p.international_minutes,
        p.roaming_minutes,
        p.sms_allowance,
        p.technology,
        p.contract_months
    FROM customers c
    JOIN subscriptions s
        ON c.customer_id = s.customer_id
    JOIN plans p
        ON s.plan_id = p.plan_id
    WHERE c.customer_id = ?
    ORDER BY s.primary_subscription_flag DESC, s.activation_date DESC;
    """
    df = pd.read_sql_query(query, conn, params=(customer_id,))
    return df_to_text(df)


def fetch_customer_churn_risk(customer_id: int) -> str:
    query = """
    SELECT 
        c.customer_id,
        c.full_name,
        ch.score_month,
        ch.churn_score,
        ch.risk_level,
        ch.main_risk_reason,
        ch.recommended_action
    FROM customer_churn_scores ch
    JOIN customers c
        ON ch.customer_id = c.customer_id
    WHERE c.customer_id = ?;
    """
    df = pd.read_sql_query(query, conn, params=(customer_id,))
    return df_to_text(df)


def fetch_customer_complaints(customer_id: int, limit: int = 5) -> str:
    query = """
    SELECT 
        complaint_date,
        complaint_category,
        complaint_description,
        severity,
        status,
        resolved_date,
        compensation_amount_jod
    FROM complaints
    WHERE customer_id = ?
    ORDER BY complaint_date DESC
    LIMIT ?;
    """
    df = pd.read_sql_query(query, conn, params=(customer_id, limit))
    return df_to_text(df)


def fetch_customer_support_interactions(customer_id: int, limit: int = 5) -> str:
    query = """
    SELECT 
        interaction_datetime,
        channel,
        reason_category,
        issue_type,
        priority,
        resolution_status,
        resolution_time_minutes,
        customer_sentiment
    FROM support_interactions
    WHERE customer_id = ?
    ORDER BY interaction_datetime DESC
    LIMIT ?;
    """
    df = pd.read_sql_query(query, conn, params=(customer_id, limit))
    return df_to_text(df)


def fetch_customer_billing_summary(customer_id: int, limit: int = 5) -> str:
    query = """
    SELECT 
        c.customer_id,
        c.full_name,
        i.invoice_id,
        i.billing_period_start,
        i.billing_period_end,
        i.issue_date,
        i.due_date,
        i.total_amount_jod,
        i.payment_status,
        i.days_overdue
    FROM customers c
    JOIN accounts a
        ON c.customer_id = a.customer_id
    JOIN invoices i
        ON a.account_id = i.account_id
    WHERE c.customer_id = ?
    ORDER BY i.issue_date DESC
    LIMIT ?;
    """
    df = pd.read_sql_query(query, conn, params=(customer_id, limit))
    return df_to_text(df)


def fetch_customer_usage_summary(customer_id: int) -> str:
    query = """
    SELECT 
        c.customer_id,
        c.full_name,
        s.subscription_id,
        s.service_type,
        COUNT(d.session_id) AS total_data_sessions,
        ROUND(SUM(d.data_used_mb) / 1024.0, 2) AS total_data_used_gb,
        ROUND(SUM(d.cost_jod), 2) AS total_data_cost_jod,
        MAX(d.session_start_time) AS last_data_session
    FROM customers c
    JOIN subscriptions s
        ON c.customer_id = s.customer_id
    LEFT JOIN data_usage_sessions d
        ON s.subscription_id = d.subscription_id
    WHERE c.customer_id = ?
    GROUP BY c.customer_id, c.full_name, s.subscription_id, s.service_type
    ORDER BY total_data_used_gb DESC;
    """
    df = pd.read_sql_query(query, conn, params=(customer_id,))
    return df_to_text(df)


def fetch_customer_value_segment(customer_id: int) -> str:
    query = """
    SELECT 
        c.customer_id,
        c.full_name,
        v.segment_month,
        v.arpu_jod,
        v.total_revenue_6m_jod,
        v.value_segment,
        v.lifetime_months
    FROM customer_value_segments v
    JOIN customers c
        ON v.customer_id = c.customer_id
    WHERE c.customer_id = ?;
    """
    df = pd.read_sql_query(query, conn, params=(customer_id,))
    return df_to_text(df)


In [ ]:
from langchain.tools import tool

@tool
def get_customer_profile(customer_id: int) -> str:
    """Get customer profile, city, segment, language, account type, and account status."""
    return fetch_customer_profile(customer_id)


@tool
def get_customer_plan(customer_id: int) -> str:
    """Get current subscriptions, plan name, monthly fee, data allowance, minutes, and contract details."""
    return fetch_customer_plan(customer_id)


@tool
def get_customer_churn_risk(customer_id: int) -> str:
    """Get churn score, risk level, main risk reason, and recommended retention action."""
    return fetch_customer_churn_risk(customer_id)


@tool
def get_customer_complaints(customer_id: int, limit: int = 5) -> str:
    """Get recent customer complaints including category, description, severity, and status."""
    return fetch_customer_complaints(customer_id, limit)


@tool
def get_customer_support_interactions(customer_id: int, limit: int = 5) -> str:
    """Get recent support interactions including channel, issue type, priority, and sentiment."""
    return fetch_customer_support_interactions(customer_id, limit)


@tool
def get_customer_billing_summary(customer_id: int, limit: int = 5) -> str:
    """Get recent invoice and billing summary including amount, payment status, and overdue days."""
    return fetch_customer_billing_summary(customer_id, limit)


@tool
def get_customer_usage_summary(customer_id: int) -> str:
    """Get customer data usage summary including total data used and last data session."""
    return fetch_customer_usage_summary(customer_id)


@tool
def get_customer_value_segment(customer_id: int) -> str:
    """Get customer value segment, ARPU, six-month revenue, and lifetime months."""
    return fetch_customer_value_segment(customer_id)


customer_tools = [
    get_customer_profile,
    get_customer_plan,
    get_customer_churn_risk,
    get_customer_complaints,
    get_customer_support_interactions,
    get_customer_billing_summary,
    get_customer_usage_summary,
    get_customer_value_segment,
]

customer_agent_prompt = """
You are a Zain Jordan customer-care AI assistant.

Use tools to retrieve customer facts.
Do not guess customer data.
Provide a structured, business-friendly answer.

For full customer analysis, include:
- Customer Summary
- Plan Summary
- Churn Risk
- Value Segment
- Billing Summary
- Complaints and Support
- Usage Summary
- Recommended Next Action
"""

customer_agent = create_agent(
    model=llm,
    tools=customer_tools,
    system_prompt=customer_agent_prompt,
)

print("Track B Tools-Based Customer Agent ready.")


## Track B Demo Question


In [ ]:
question = """
Analyze customer 42.
Check profile, plan, churn risk, value segment, billing, complaints, support history, and usage.
Recommend the next best action.
"""

print(run_agent(customer_agent, question))


# Track C: RAG Starter

Use this if your project needs plan recommendations, add-on recommendations, campaign suggestions, complaint themes, or customer experience insights.


In [ ]:
from langchain_core.documents import Document

def build_plan_documents(conn):
    df = pd.read_sql_query("""
    SELECT 
        plan_id,
        plan_name,
        plan_category,
        service_type,
        monthly_fee_jod,
        data_allowance_gb,
        local_minutes,
        international_minutes,
        roaming_minutes,
        sms_allowance,
        technology,
        contract_months,
        data_carryover_flag,
        is_business_plan,
        status
    FROM plans
    WHERE status = 'Active';
    """, conn)

    docs = []
    for _, row in df.iterrows():
        text = f"""
Plan ID: {row['plan_id']}
Plan Name: {row['plan_name']}
Category: {row['plan_category']}
Service Type: {row['service_type']}
Monthly Fee: {row['monthly_fee_jod']} JOD
Data Allowance: {row['data_allowance_gb']} GB
Local Minutes: {row['local_minutes']}
International Minutes: {row['international_minutes']}
Roaming Minutes: {row['roaming_minutes']}
SMS Allowance: {row['sms_allowance']}
Technology: {row['technology']}
Contract Months: {row['contract_months']}
Business Plan: {bool(row['is_business_plan'])}
Use this plan for plan recommendation and customer-care scenarios.
"""
        docs.append(Document(
            page_content=text.strip(),
            metadata={"source_table": "plans", "row_id": int(row["plan_id"]), "document_type": "plan"}
        ))
    return docs


def build_addon_documents(conn):
    df = pd.read_sql_query("""
    SELECT 
        addon_id,
        addon_name,
        addon_type,
        price_jod,
        validity_days,
        data_gb,
        minutes,
        sms,
        technology
    FROM addons;
    """, conn)

    docs = []
    for _, row in df.iterrows():
        text = f"""
Add-on ID: {row['addon_id']}
Add-on Name: {row['addon_name']}
Add-on Type: {row['addon_type']}
Price: {row['price_jod']} JOD
Validity Days: {row['validity_days']}
Data: {row['data_gb']} GB
Minutes: {row['minutes']}
SMS: {row['sms']}
Technology: {row['technology']}
Use this add-on for data, roaming, voice, SMS, or technology-specific recommendation.
"""
        docs.append(Document(
            page_content=text.strip(),
            metadata={"source_table": "addons", "row_id": int(row["addon_id"]), "document_type": "addon"}
        ))
    return docs


def build_campaign_documents(conn):
    df = pd.read_sql_query("""
    SELECT 
        campaign_id,
        campaign_name,
        campaign_type,
        start_date,
        end_date,
        target_segment,
        offer_description,
        channel
    FROM campaigns;
    """, conn)

    docs = []
    for _, row in df.iterrows():
        text = f"""
Campaign ID: {row['campaign_id']}
Campaign Name: {row['campaign_name']}
Campaign Type: {row['campaign_type']}
Target Segment: {row['target_segment']}
Offer Description: {row['offer_description']}
Channel: {row['channel']}
Use this campaign for promotion, targeting, and customer engagement recommendations.
"""
        docs.append(Document(
            page_content=text.strip(),
            metadata={"source_table": "campaigns", "row_id": int(row["campaign_id"]), "document_type": "campaign"}
        ))
    return docs


def build_experience_documents(conn, limit=150):
    complaint_df = pd.read_sql_query("""
    SELECT complaint_id, customer_id, complaint_category, complaint_description, severity, status
    FROM complaints
    ORDER BY complaint_date DESC
    LIMIT ?;
    """, conn, params=(limit,))

    support_df = pd.read_sql_query("""
    SELECT interaction_id, customer_id, channel, reason_category, issue_type, priority, customer_sentiment
    FROM support_interactions
    ORDER BY interaction_datetime DESC
    LIMIT ?;
    """, conn, params=(limit,))

    docs = []

    for _, row in complaint_df.iterrows():
        text = f"""
Complaint ID: {row['complaint_id']}
Customer ID: {row['customer_id']}
Category: {row['complaint_category']}
Description: {row['complaint_description']}
Severity: {row['severity']}
Status: {row['status']}
Use this complaint to understand customer pain points and experience risks.
"""
        docs.append(Document(
            page_content=text.strip(),
            metadata={"source_table": "complaints", "row_id": int(row["complaint_id"]), "document_type": "complaint"}
        ))

    for _, row in support_df.iterrows():
        text = f"""
Support Interaction ID: {row['interaction_id']}
Customer ID: {row['customer_id']}
Channel: {row['channel']}
Reason Category: {row['reason_category']}
Issue Type: {row['issue_type']}
Priority: {row['priority']}
Sentiment: {row['customer_sentiment']}
Use this support interaction to understand support patterns and service issues.
"""
        docs.append(Document(
            page_content=text.strip(),
            metadata={"source_table": "support_interactions", "row_id": int(row["interaction_id"]), "document_type": "support_interaction"}
        ))

    return docs


rag_documents = (
    build_plan_documents(conn)
    + build_addon_documents(conn)
    + build_campaign_documents(conn)
    + build_experience_documents(conn, limit=150)
)

print("RAG documents:", len(rag_documents))
Counter(doc.metadata["document_type"] for doc in rag_documents)


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100
)

split_docs = text_splitter.split_documents(rag_documents)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = InMemoryVectorStore(embeddings)
vector_store.add_documents(split_docs)

retriever = vector_store.as_retriever(search_kwargs={"k": 5})

print("Track C RAG retriever ready. Chunks:", len(split_docs))


In [ ]:
def format_docs(docs):
    formatted = []
    for i, doc in enumerate(docs, start=1):
        formatted.append(
            f"[Document {i} | Source: {doc.metadata.get('source_table')} | Type: {doc.metadata.get('document_type')} | Row ID: {doc.metadata.get('row_id')}]\n{doc.page_content}"
        )
    return "\n\n".join(formatted)


def rag_answer(question: str, k: int = 5) -> str:
    docs = retriever.invoke(question)[:k]
    context = format_docs(docs)

    prompt = f"""
You are a telecom recommendation assistant for Zain Jordan.

Answer using only the retrieved context.

Rules:
- Do not invent facts.
- If context is insufficient, say what is missing.
- Provide business-friendly recommendations.
- Include sources used.

Retrieved Context:
{context}

Question:
{question}

Answer:
"""
    response = llm.invoke(prompt)
    return response.content


@tool
def search_telecom_knowledge(question: str) -> str:
    """Search telecom plans, add-ons, campaigns, complaints, and support knowledge using RAG."""
    return rag_answer(question)


rag_agent = create_agent(
    model=llm,
    tools=[search_telecom_knowledge],
    system_prompt="""
You are a Zain Jordan RAG recommendation agent.
Use the search_telecom_knowledge tool for recommendations about plans, add-ons, campaigns, complaints, and support themes.
Do not guess. Use retrieved context.
"""
)

print("Track C RAG Agent ready.")


## Track C Demo Question


In [ ]:
question = "Recommend a plan, add-on, or campaign for a customer with high data usage and churn risk."

print(run_agent(rag_agent, question))


# Track D: Multi-Agent Starter

Use this if your project needs multiple specialist roles.


In [ ]:
def care_message_response(context: str) -> str:
    prompt = f"""
You are a professional Zain Jordan care-message assistant.

Draft:
1. A short customer-facing message
2. A short internal follow-up note

Rules:
- Do not mention internal churn score directly to the customer.
- Be polite, professional, and clear.
- Do not overpromise compensation.
- Keep customer message under 120 words.

Context:
{context}

Output:
"""
    return llm.invoke(prompt).content


@tool
def customer_agent_tool(task: str) -> str:
    """Use the customer agent to retrieve customer-specific profile, plan, churn, billing, complaints, support, usage, and value details."""
    return run_agent(customer_agent, task)


@tool
def sql_agent_tool(question: str) -> str:
    """Use the SQL agent to answer broad telecom business questions from the database."""
    return run_agent(sql_agent, question)


@tool
def rag_agent_tool(question: str) -> str:
    """Use the RAG agent to recommend plans, add-ons, campaigns, or explain complaint/support themes."""
    return run_agent(rag_agent, question)


@tool
def care_message_tool(context: str) -> str:
    """Draft a professional customer-facing message and internal follow-up note."""
    return care_message_response(context)


supervisor_agent = create_agent(
    model=llm,
    tools=[customer_agent_tool, sql_agent_tool, rag_agent_tool, care_message_tool],
    system_prompt="""
You are the Supervisor Agent for a Zain Jordan capstone AI system.

Coordinate specialist tools:
- customer_agent_tool for customer-specific facts
- sql_agent_tool for broader business analytics
- rag_agent_tool for plans, campaigns, add-ons, and experience recommendations
- care_message_tool for professional communication

Do not invent facts.
Use tools when needed.
Return a structured final answer with:
1. Summary
2. Evidence from data
3. Recommendation
4. Customer message or internal note
"""
)

print("Track D Multi-Agent Supervisor ready.")


## Track D Demo Question


In [ ]:
question = """
Analyze customer 42.
Check profile, churn risk, billing, complaints, support, usage, and value.
Recommend a relevant plan, add-on, or campaign.
Draft a professional customer-care message.
"""

print(run_agent(supervisor_agent, question))


# Track E: MCP-Enhanced Starter

Use this only if your team wants to show enterprise-style tool exposure.

This section creates a small MCP server with three tools:

1. `get_customer_profile`
2. `get_customer_churn_risk`
3. `get_customer_value_segment`

Recommended for confident teams only.


In [ ]:
MCP_SERVER_FILE = Path("capstone_mcp_server.py").resolve()

mcp_server_lines = [
    "import sqlite3",
    "import pandas as pd",
    "from pathlib import Path",
    "",
    "try:",
    "    from fastmcp import FastMCP",
    "except ImportError:",
    "    from mcp.server.fastmcp import FastMCP",
    "",
    f"DB_PATH = Path({repr(str(db_path_obj))})",
    "",
    "mcp = FastMCP('Zain Jordan Capstone MCP Server')",
    "",
    "def get_connection():",
    "    return sqlite3.connect(str(DB_PATH), check_same_thread=False)",
    "",
    "def df_to_text(df, max_rows=10):",
    "    if df is None or df.empty:",
    "        return 'No records found.'",
    "    return df.head(max_rows).to_string(index=False)",
    "",
    "@mcp.tool()",
    "def get_customer_profile(customer_id: int) -> str:",
    "    '''Get customer profile for a Zain Jordan customer ID.'''",
    "    conn = get_connection()",
    "    query = '''",
    "    SELECT customer_id, full_name, city, customer_segment, preferred_language, status",
    "    FROM customers",
    "    WHERE customer_id = ?;",
    "    '''",
    "    df = pd.read_sql_query(query, conn, params=(customer_id,))",
    "    conn.close()",
    "    return df_to_text(df)",
    "",
    "@mcp.tool()",
    "def get_customer_churn_risk(customer_id: int) -> str:",
    "    '''Get churn risk details for a Zain Jordan customer ID.'''",
    "    conn = get_connection()",
    "    query = '''",
    "    SELECT c.customer_id, c.full_name, ch.churn_score, ch.risk_level, ch.main_risk_reason, ch.recommended_action",
    "    FROM customer_churn_scores ch",
    "    JOIN customers c ON ch.customer_id = c.customer_id",
    "    WHERE c.customer_id = ?;",
    "    '''",
    "    df = pd.read_sql_query(query, conn, params=(customer_id,))",
    "    conn.close()",
    "    return df_to_text(df)",
    "",
    "@mcp.tool()",
    "def get_customer_value_segment(customer_id: int) -> str:",
    "    '''Get customer value segment, ARPU, and six-month revenue.'''",
    "    conn = get_connection()",
    "    query = '''",
    "    SELECT c.customer_id, c.full_name, v.arpu_jod, v.total_revenue_6m_jod, v.value_segment",
    "    FROM customer_value_segments v",
    "    JOIN customers c ON v.customer_id = c.customer_id",
    "    WHERE c.customer_id = ?;",
    "    '''",
    "    df = pd.read_sql_query(query, conn, params=(customer_id,))",
    "    conn.close()",
    "    return df_to_text(df)",
    "",
    "if __name__ == '__main__':",
    "    mcp.run(transport='stdio')",
]

MCP_SERVER_FILE.write_text("\n".join(mcp_server_lines), encoding="utf-8")
print("MCP server file created:", MCP_SERVER_FILE)


In [ ]:
# Optional MCP client setup.
# This cell uses top-level await, which works in Google Colab/Jupyter.
# Run this only if your team chooses Track E.

from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_client = MultiServerMCPClient(
    {
        "capstone_zain": {
            "transport": "stdio",
            "command": "python",
            "args": [str(MCP_SERVER_FILE)],
        }
    }
)

mcp_tools = await mcp_client.get_tools()

print("MCP tools loaded:")
for t in mcp_tools:
    print("-", t.name)

mcp_agent = create_agent(
    model="openai:gpt-4.1-mini",
    tools=mcp_tools,
    system_prompt="""
You are a Zain Jordan MCP-powered capstone assistant.
Use MCP tools to retrieve customer profile, churn risk, and value segment.
Do not guess facts.
"""
)

print("Track E MCP Agent ready.")


## Track E Demo Question


In [ ]:
# Run only if Track E setup completed.

mcp_result = await mcp_agent.ainvoke({
    "messages": [
        {"role": "user", "content": "Use MCP tools to analyze customer 42 profile, churn risk, and value segment."}
    ]
})

print(extract_final_text(mcp_result))


# Final Demo Output Template


In [ ]:
final_demo_prompt = team_config["main_demo_prompt"]

print("Final Demo Prompt:")
print(final_demo_prompt)

# Choose one path below based on your project.
# Uncomment the option your team is using.

# Option A: SQL Agent
# final_answer = run_agent(sql_agent, final_demo_prompt)

# Option B: Customer Tools Agent
# final_answer = run_agent(customer_agent, final_demo_prompt)

# Option C: RAG Agent
# final_answer = run_agent(rag_agent, final_demo_prompt)

# Option D: Multi-Agent Supervisor
final_answer = run_agent(supervisor_agent, final_demo_prompt)

print(final_answer)


# Save Final Demo Output


In [ ]:
report = f"""
# Capstone Demo Output

Generated: {datetime.now().strftime("%Y-%m-%d %H:%M")}

## Team

{team_config["team_name"]}

## Project Title

{team_config["project_title"]}

## Business Problem

{team_config["business_problem"]}

## Target User

{team_config["target_user"]}

## Selected Track

{team_config["selected_track"]}

## Database Tables

{", ".join(team_config["database_tables"])}

## Main Demo Prompt

{final_demo_prompt}

## Final AI Output

{final_answer}
"""

output_path = save_text_report("capstone_demo_output.md", report)


# Final Presentation Checklist

Before final presentation, confirm:

1. Project title is clear.
2. Business problem is specific.
3. Target user is defined.
4. Data tables are listed.
5. Architecture is explained simply.
6. Demo prompt works.
7. Output is database-backed.
8. Recommendation is useful.
9. Team can explain limitations.
10. Team has future improvements.


In [ ]:
presentation_checklist = pd.DataFrame([
    {"item": "Project title is clear", "status": "Not Checked"},
    {"item": "Business problem is specific", "status": "Not Checked"},
    {"item": "Target user is defined", "status": "Not Checked"},
    {"item": "Database tables are listed", "status": "Not Checked"},
    {"item": "Architecture is explained simply", "status": "Not Checked"},
    {"item": "Demo prompt works", "status": "Not Checked"},
    {"item": "Output is database-backed", "status": "Not Checked"},
    {"item": "Recommendation is useful", "status": "Not Checked"},
    {"item": "Limitations are explained", "status": "Not Checked"},
    {"item": "Future improvements are ready", "status": "Not Checked"},
])

presentation_checklist


# Trainer Build Sprint Checkpoints

## Checkpoint 1: Scope Check

- Is the project too broad?
- Is the target user clear?
- Is the demo prompt specific?

## Checkpoint 2: Data Check

- Are the required tables available?
- Does the team know which fields matter?
- Is the output grounded in the database?

## Checkpoint 3: Architecture Check

- Are they using the right pattern?
- SQL for analytics?
- RAG for recommendations?
- Tools/MCP for customer lookup?
- Multi-agent only if needed?

## Checkpoint 4: Demo Check

- Does the notebook run?
- Is the answer understandable?
- Is the recommendation useful?
- Can the team present in 5 minutes?


# Trainer Closing Script

Today, teams moved from planning to building.

The goal was not to build a perfect production system.

The goal was to build a working, database-backed AI prototype that solves one clear telecom business problem.

In the next class, teams will present:

1. Business problem
2. Target user
3. Data used
4. AI architecture
5. Live demo
6. Business value
7. Future improvements
